# Iramuteq Converter

In [9]:
import pandas as pd
import nltk

# Ensure punkt and stopwords are downloaded for tokenization
nltk.download('punkt', quiet=False, force=True)
nltk.download('stopwords', quiet=False, force=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Re-read the CSV file with correct separator and header (semicolon separator)
df = pd.read_csv('data-uc3m25.csv', sep=';', header=0, on_bad_lines='skip')

# Now select the required columns
df_filtered = df[['m.id', 'm.session', 'm.question', 'm.creation_date']].copy()

df_filtered = df_filtered.groupby('m.session').filter(lambda x: x['m.id'].nunique() > 1)

# remove spanish stop words
stop_words = set(stopwords.words('spanish'))
# remove english stop words
stop_words.update(set(stopwords.words('english')))

stop_words.update(['hola', 'mucha', 'gracias', 'por', 'de', 'la', 'el', 'y', 'a', 'en', 'que', 'con', 'los', 'las', 'un', 'una'])

# Function to remove stop words from a text
def remove_stopwords(text):
    word_tokens = word_tokenize(str(text).lower(), language='spanish')
    filtered_text = ' '.join([word for word in word_tokens if word not in stop_words])
    return filtered_text

# Apply the function to the 'm.question' column
df_filtered.loc[:, 'm.question'] = df_filtered['m.question'].astype(str).apply(remove_stopwords)


[nltk_data] Downloading package punkt to /Users/rpm/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /Users/rpm/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Convert CSV to Iramuteq format

```python

In [ ]:
# Converter df_filtered para formato Iramuteq com m.id sequencial por sessão
# m.id será sequencial dentro de cada m.session, começando em 1

required_columns = ['m.session', 'm.creation_date', 'm.question']

df_filtered = df_filtered.copy()
# Adiciona coluna m.id sequencial por sessão
df_filtered['m.id'] = df_filtered.groupby('m.session').cumcount() + 1

with open('uc3m_iramuteq.txt', 'w', encoding='utf-8') as f:
    for idx, row in df_filtered.iterrows():
        header = (
            f"****session={row['m.session']} "
            f"id={row['m.id']}"
        )
        texto = str(row['m.question'])
        f.write(f"{header}\n{texto}\n\n")

print('File uc3m_iramuteq.txt created')


File uc3m_iramuteq.txt created
